# Manhattan Skyline Explorer — interactive 3D map + construction timeline

**Run in Google Colab:** `Runtime → Run all` (or run the cells top to bottom). The notebook downloads real Manhattan building records from [NYC Open Data — BUILDING_P](https://data.cityofnewyork.us/d/u9wf-3gbt), cleans them, and creates an **interactive skyline dashboard** on top of OpenStreetMap-derived tiles.

**Inside the interactive dashboard:**
- **Year slider:** show tall buildings completed *by* that year, or completed *during* that exact year.
- **3D/2D buttons:** show physical height as an extruded column (3D) or a size-scaled point (2D).
- **Minimum height** and **height exaggeration** controls.
- **Top-height bar chart:** click a building to fly to its location; hover on a map column for coordinates, height, and completion year.
- **Decade chart:** click a decade to jump through skyline history.
- **Export CSV** from the dashboard or the Python notebook.

**Important:** The visualization uses NYC's mapped **roof height** (feet above the building's ground level), not architectural height including spires. Missing year/height records are excluded. Construction years may be approximate, and the historical slider filters *today's standing buildings*—it does **not** reconstruct demolished buildings. Most records do not have a commonly known name. See the final section for caveats and sources.

In [ ]:
# Usually already installed in Colab; safe to rerun.
%pip -q install pandas requests nbformat

In [ ]:
import json
import html
from datetime import date
from pathlib import Path

import pandas as pd
import requests
from IPython.display import HTML, display

DATASET_ID = "u9wf-3gbt"  # Official NYC building-centroid layer
API_URL = f"https://data.cityofnewyork.us/resource/{DATASET_ID}.json"
MIN_DOWNLOAD_HEIGHT_FT = 100  # Download broadly; browser slider can narrow it
EARLIEST_YEAR = 1850
LATEST_YEAR = date.today().year
PAGE_SIZE = 30000

# A Manhattan building's BIN starts with 1; keep actual buildings only.
WHERE = (
    "bin >= 1000000 AND bin < 2000000 "
    "AND feature_code = 2100 "
    f"AND height_roof >= {MIN_DOWNLOAD_HEIGHT_FT} "
    f"AND construction_year BETWEEN {EARLIEST_YEAR} AND {LATEST_YEAR}"
)
print("Official NYC dataset:", f"https://data.cityofnewyork.us/d/{DATASET_ID}")
print("Minimum downloaded height:", MIN_DOWNLOAD_HEIGHT_FT, "ft")

In [ ]:
def fetch_nyc_buildings():
    """Fetch the point centroids plus height and year, paginating if necessary."""
    rows = []
    session = requests.Session()
    session.headers.update({"User-Agent": "ManhattanSkylineExplorer/1.0 (academic project)"})
    offset = 0
    while True:
        params = {
            "$select": "the_geom,name,bin,doitt_id,objectid,construction_year,height_roof,feature_code",
            "$where": WHERE,
            "$order": "objectid ASC",
            "$limit": PAGE_SIZE,
            "$offset": offset,
        }
        try:
            response = session.get(API_URL, params=params, timeout=90)
            response.raise_for_status()
            page = response.json()
            if not isinstance(page, list):
                raise ValueError(f"Unexpected API response: {str(page)[:400]}")
        except (requests.RequestException, ValueError) as exc:
            raise RuntimeError(
                "NYC Open Data could not be fetched. Check internet access and rerun this cell; "
                f"the API is {API_URL}. Original error: {exc}"
            ) from exc
        rows.extend(page)
        print(f"Retrieved {len(rows):,} records...", end="\r")
        if len(page) < PAGE_SIZE:
            break
        offset += PAGE_SIZE
    print(f"Downloaded {len(rows):,} records from NYC Open Data.     ")
    return rows

raw_records = fetch_nyc_buildings()


def clean_buildings(raw_records):
    clean = []
    for r in raw_records:
        geom = r.get("the_geom") or {}
        coords = geom.get("coordinates") or []
        if len(coords) < 2:
            continue
        try:
            lon, lat = float(coords[0]), float(coords[1])
            height_ft = float(r["height_roof"])
            year = int(float(r["construction_year"]))
            bin_id = str(int(float(r.get("bin", 0))))
        except (TypeError, ValueError, KeyError):
            continue
        # Sanity-check centroid coordinates, measured roof height and valid year.
        if not (-74.06 <= lon <= -73.85 and 40.68 <= lat <= 40.89):
            continue
        if not (MIN_DOWNLOAD_HEIGHT_FT <= height_ft <= 3000):
            continue
        if not (EARLIEST_YEAR <= year <= LATEST_YEAR):
            continue
        name = (r.get("name") or "").strip()
        building_id = str(r.get("doitt_id") or r.get("objectid") or bin_id)
        clean.append({
            "id": building_id,
            "name": name if name else f"Building BIN {bin_id}",
            "named": bool(name),
            "bin": bin_id,
            "year": year,
            "height_ft": round(height_ft, 1),
            "height_m": round(height_ft * 0.3048, 1),
            "lat": round(lat, 6),
            "lon": round(lon, 6),
        })
    return pd.DataFrame(clean).drop_duplicates(subset=["id", "lat", "lon"])

buildings = clean_buildings(raw_records)
if buildings.empty:
    raise ValueError("No buildings were found. Check dataset access or filters above.")
buildings = buildings.sort_values("height_ft", ascending=False).reset_index(drop=True)
print(f"Usable Manhattan buildings ≥ {MIN_DOWNLOAD_HEIGHT_FT} ft: {len(buildings):,}")
print(f"Mapped years: {buildings.year.min()}–{buildings.year.max()}")
print(f"Names available: {buildings.named.sum():,} / {len(buildings):,}")
display(buildings.head(12))

## Interactive map and dashboard

The next cell builds a **standalone HTML dashboard** with its own JavaScript controls (not dependent on Colab widget syncing). A browser preview appears directly below it. The HTML is also saved to `/content/manhattan_skyline_dashboard.html` in Colab so you can open it full screen or share it.

Use **3D Columns** + mouse drag to tilt/rotate; the bars represent actual mapped height (or exaggerated height if selected). Click an item in **Tallest visible buildings** to zoom to it. Move the year slider to see the skyline evolve.

In [ ]:
HTML_TEMPLATE = r'''<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1"/>
<title>Manhattan Skyline Explorer</title>
<script src="https://unpkg.com/deck.gl@^9.0.0/dist.min.js"></script>
<style>
*{box-sizing:border-box}html,body{margin:0;height:100%;font-family:Inter,system-ui,-apple-system,Segoe UI,sans-serif;color:#152238;background:#f7f9fb}
#shell{height:100vh;min-height:600px;display:grid;grid-template-columns:minmax(0,1fr) 340px;overflow:hidden}
#left{position:relative;min-width:0;min-height:0;background:#dae8eb}#map{position:absolute;inset:0}#map canvas{outline:none}
#mast{position:absolute;top:18px;left:20px;z-index:5;padding:13px 17px;background:rgba(255,255,255,.94);backdrop-filter:blur(8px);border-radius:12px;box-shadow:0 3px 24px #163a4c24;pointer-events:none}
#mast .eyebrow{letter-spacing:.13em;font-size:10px;font-weight:800;color:#07827c}#mast h1{font-size:22px;margin:3px 0 0;font-weight:850;letter-spacing:-.04em}#mast p{font-size:11px;margin:4px 0 0;color:#5e7180}
#legend{position:absolute;bottom:15px;left:17px;background:#fffffff0;border-radius:9px;padding:10px 13px;z-index:5;box-shadow:0 3px 20px #18394d1c;font-size:11px}
.legrow{display:flex;align-items:center;gap:7px;margin-top:6px}.swatch{width:9px;height:9px;border-radius:3px}
#mapHelp{position:absolute;right:15px;bottom:15px;background:#ffffffde;padding:9px 11px;border-radius:8px;font-size:11px;color:#42566b;z-index:5}
#side{background:white;border-left:1px solid #dce5ed;display:flex;flex-direction:column;overflow-y:auto;box-shadow:-4px 0 20px #202d3d13;z-index:6}
#sideInner{padding:21px 21px 12px}h2{font-size:11px;letter-spacing:.1em;text-transform:uppercase;color:#78899a;margin:18px 0 9px}
#yearHeading{display:flex;justify-content:space-between;align-items:baseline;gap:10px}#year{font-weight:850;font-size:40px;letter-spacing:-.07em;color:#15304b}#modeNote{font-size:10px;color:#6d7c8d}
input[type=range]{width:100%;accent-color:#008f86;cursor:pointer}#yearTicks{display:flex;justify-content:space-between;color:#82929f;font-size:10px;margin-top:1px}
button,select,input[type=search]{font:inherit}button{cursor:pointer;border:1px solid #d7e0e8;border-radius:8px;background:#f8fafc;color:#314355;padding:8px 12px;font-size:12px;font-weight:650}
button.active{background:#096e70;color:white;border-color:#096e70}button:focus-visible,input:focus-visible,select:focus-visible{outline:3px solid #69d7c6;outline-offset:2px}
.toggle{display:flex;gap:5px;flex-wrap:wrap}.toggle button{flex:1;min-width:84px}#playButton{margin-top:9px;width:100%;background:#f0f8f6;border:1px solid #9bd0c7;color:#07776f}
.inline{display:flex;align-items:center;justify-content:space-between;gap:10px}.inline label{font-size:12px;color:#58697c}.value{font-weight:700;color:#0a7573;font-size:12px}
select,input[type=search]{border:1px solid #d9e3e9;border-radius:8px;background:white;color:#324c60;padding:8px;font-size:12px;width:100%}
#stats{display:grid;grid-template-columns:1fr 1fr;gap:9px;margin-top:17px}.stat{padding:11px;border:1px solid #e2eaee;border-radius:10px;background:#f8fafb}.stat strong{display:block;font-size:23px;letter-spacing:-.05em;color:#173950}.stat small{font-size:10px;color:#697e8b}
#rankings{min-height:90px}.barrow{display:grid;grid-template-columns:1fr 62px;gap:6px;align-items:center;width:100%;background:transparent;border:0;border-radius:7px;padding:6px 5px;text-align:left}.barrow:hover{background:#f1f8f6}.barname{font-size:11px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;color:#415368}.barval{font-size:11px;text-align:right;font-weight:800;color:#126f74}.barbg{height:6px;margin-top:5px;background:#e9eff2;border-radius:8px;overflow:hidden}.barfill{height:100%;border-radius:8px;background:linear-gradient(90deg,#34aaa3,#106d86)}
#decades{display:flex;align-items:flex-end;height:73px;gap:3px;padding-bottom:17px;border-bottom:1px solid #e2e8ed}.decbar{flex:1;min-width:5px;background:#55b4aa;border-radius:3px 3px 0 0;position:relative;cursor:pointer}.decbar:hover{background:#e89d57}.decbar:nth-child(4n):after{content:attr(data-decade);position:absolute;top:100%;left:-5px;font-size:9px;color:#8696a2}
#footer{font-size:10px;color:#8995a0;padding:12px 21px 17px;line-height:1.5}.primary{width:100%;background:#12374d;color:white;border-color:#12374d}
@media(max-width:730px){#shell{display:flex;flex-direction:column;overflow:auto;height:auto}#left{height:57vh;min-height:400px;flex:none}#side{overflow:visible}#mast{left:10px;top:9px;padding:9px 11px}#mast h1{font-size:18px}#mapHelp{display:none}}
</style></head><body>
<div id="shell"><section id="left"><div id="map"></div>
 <div id="mast"><div class="eyebrow">NYC OPEN DATA · SKYLINE ATLAS</div><h1>Manhattan in time</h1><p>Explore buildings · completion years · real roof heights</p></div>
 <div id="legend"><strong>Completion year</strong><div class="legrow"><span class="swatch" style="background:#286d88"></span>Before 1930</div><div class="legrow"><span class="swatch" style="background:#419f9d"></span>1930–1969</div><div class="legrow"><span class="swatch" style="background:#7269b2"></span>1970–1999</div><div class="legrow"><span class="swatch" style="background:#eeab5d"></span>2000–2014</div><div class="legrow"><span class="swatch" style="background:#d66d5f"></span>2015 onward</div></div>
 <div id="mapHelp">Drag to pan · scroll to zoom · Ctrl-drag to rotate/tilt</div>
 </section><aside id="side"><div id="sideInner">
 <div class="inline"><span style="font-size:12px;color:#607486;font-weight:700">CONSTRUCTION TIMELINE</span><span id="modeNote">Built by selected year</span></div>
 <div id="yearHeading"><div id="year">2026</div><div style="font-size:11px;color:#647c8a">Year completed</div></div>
 <input id="yearSlider" aria-label="Construction year" type="range" min="1850" max="2026" step="1" value="2026"/>
 <div id="yearTicks"><span id="minYearLabel">1850</span><span id="maxYearLabel">2026</span></div>
 <div class="toggle" style="margin-top:12px"><button id="cumulative" class="active">Built by year</button><button id="exact">Only in year</button></div>
 <button id="playButton">▶ Play skyline growth (+5 years / frame)</button>
 <h2>Map appearance</h2>
 <div class="toggle"><button class="active" id="threeD">3D columns</button><button id="twoD">2D bubbles</button></div>
 <div class="inline" style="margin-top:13px"><label for="heightSlider">Minimum roof height</label><span class="value" id="heightLabel">150 ft</span></div>
 <input id="heightSlider" aria-label="Minimum height feet" type="range" min="100" max="1000" step="50" value="150"/>
 <div class="inline" style="margin:10px 0"><label for="scaleSelect">3D height scaling</label><select id="scaleSelect" style="width:110px"><option value="1">1× actual</option><option value="2">2×</option><option value="4">4×</option></select></div>
 <input id="search" type="search" placeholder="Find by name or BIN (map + chart)" aria-label="Search buildings"/>
 <div id="stats"><div class="stat"><strong id="count">0</strong><small>Buildings in selection</small></div><div class="stat"><strong id="maxHeight">—</strong><small>Highest roof (ft)</small></div></div>
 <h2>12 tallest visible buildings · click to zoom</h2><div id="rankings"></div>
 <h2>Completion decade · click to jump</h2><div id="decades"></div>
 <button class="primary" id="exportCSV" style="margin-top:17px">↓ Export selected records · CSV</button>
 </div><div id="footer">Map tiles © OpenStreetMap contributors, © CARTO. Heights measured to roofs (feet); spires excluded. A year filter is not a reconstruction of demolitions. NYC Open Data BUILDING_P (u9wf-3gbt). Generated __GENERATED__.</div></aside></div>
<script>
const BUILDINGS = __DATA__;
const YEAR_MIN = __MINYEAR__, YEAR_MAX = __MAXYEAR__;
const $ = id => document.getElementById(id);
$('yearSlider').min = YEAR_MIN; $('yearSlider').max = YEAR_MAX;
$('yearSlider').value = YEAR_MAX; $('minYearLabel').textContent = YEAR_MIN;
$('maxYearLabel').textContent = YEAR_MAX;
let selectedYear = YEAR_MAX, minHeight = 150, view3D = true, cumulative = true, scale3D = 1;
let searchText = '', shown = [], playback = null;
function colorForYear(y){
  return y < 1930 ? [40,109,136,220] : y < 1970 ? [65,159,157,220] : y < 2000 ? [114,105,178,225] : y < 2015 ? [238,171,93,230] : [214,109,95,234];
}
function escapeHtml(s){return String(s).replace(/[&<>"']/g, c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));}
function label(d){return d.name.startsWith('Building BIN ') ? `BIN ${d.bin}` : d.name;}
function filtered(){
  return BUILDINGS.filter(d => d.height_ft >= minHeight &&
      (cumulative ? d.year <= selectedYear : d.year === selectedYear) &&
      (!searchText || (d.name + ' ' + d.bin).toLowerCase().includes(searchText)));
}
const tiles = new deck.TileLayer({
  id: 'carto-osm-tiles', data: 'https://a.basemaps.cartocdn.com/light_all/{z}/{x}/{y}.png',
  minZoom: 0, maxZoom: 19, pickable: false,
  renderSubLayers: props => {
    const b = props.tile.boundingBox;
    return new deck.BitmapLayer(props, {
      id: `${props.id}-bitmap`, data: null, image: props.data,
      bounds: [b[0][0], b[0][1], b[1][0], b[1][1]], pickable: false
    });
  }
});
const atlas = new deck.Deck({
  parent: $('map'), initialViewState: {latitude:40.764, longitude:-73.984, zoom:11.7, pitch:54, bearing:-18},
  controller: true, useDevicePixels: true, layers: [tiles],
  getTooltip: ({object}) => object && object.height_ft ? {
    html: `<strong>${escapeHtml(label(object))}</strong><br/>Completed: <b>${object.year}</b><br/>Roof: <b>${object.height_ft.toLocaleString()} ft</b> (${object.height_m} m)<br/>Latitude: ${object.lat}<br/>Longitude: ${object.lon}<br/>BIN: ${escapeHtml(object.bin)}`,
    style: {backgroundColor:'#143047', color:'white', borderRadius:'9px', padding:'11px 13px', fontSize:'12px', lineHeight:'1.7', maxWidth:'270px'}
  } : null,
  onClick: ({object}) => {if (object && object.height_ft) flyTo(object);}
});
function flyTo(d){
  atlas.setProps({initialViewState: {
    longitude:d.lon, latitude:d.lat, zoom:15.7, pitch:view3D ? 57 : 0, bearing:-18,
    transitionDuration:1000, transitionInterpolator:new deck.FlyToInterpolator()
  }});
}
function renderMap(){
  const marks = view3D ? new deck.ColumnLayer({
    id:'buildings-3d', data:shown, pickable:true, extruded:true,
    diskResolution:9, radius:22, getPosition:d=>[d.lon,d.lat],
    getElevation:d=>d.height_m, elevationScale:scale3D,
    getFillColor:d=>colorForYear(d.year), opacity:.92, material:{ambient:.45,diffuse:.58,shininess:32,specularColor:[190,190,190]},
    autoHighlight:true, highlightColor:[30,140,136,125]
  }) : new deck.ScatterplotLayer({
    id:'buildings-2d', data:shown, pickable:true, filled:true, stroked:true,
    getPosition:d=>[d.lon,d.lat], getRadius:d=>Math.max(18,Math.min(85,d.height_ft/12)),
    radiusMinPixels:3, radiusMaxPixels:22, getFillColor:d=>colorForYear(d.year),
    getLineColor:[255,255,255,240], getLineWidth:1, lineWidthMinPixels:1, autoHighlight:true
  });
  atlas.setProps({layers:[tiles,marks]});
}
function renderRanks(){
  const arr = [...shown].sort((a,b)=>b.height_ft-a.height_ft).slice(0,12);
  const max = arr.length ? arr[0].height_ft : 1;
  $('rankings').innerHTML = arr.length ? arr.map((d,i)=>
    `<button class="barrow" data-id="${escapeHtml(d.id)}" title="Zoom to ${escapeHtml(label(d))}"><span><span class="barname" style="display:block">${i+1}. ${escapeHtml(label(d))}</span><span class="barbg" style="display:block"><span class="barfill" style="display:block;width:${(100*d.height_ft/max).toFixed(1)}%"></span></span></span><span class="barval">${Math.round(d.height_ft).toLocaleString()} ft</span></button>`
  ).join('') : '<p style="font-size:12px;color:#889aa7">No matching buildings. Try a different year or height.</p>';
  $('rankings').querySelectorAll('.barrow').forEach((b,i)=>b.addEventListener('click',()=>flyTo(arr[i])));
}
function renderDecades(){
  const counts = new Map();
  // Plot completed buildings at the chosen height threshold, not just year-specific matches.
  for (const d of BUILDINGS){
    if(d.height_ft<minHeight || (searchText && !(d.name+' '+d.bin).toLowerCase().includes(searchText))) continue;
    const decade = Math.floor(d.year/10)*10;
    counts.set(decade, (counts.get(decade)||0)+1);
  }
  const labels = []; for(let d=Math.floor(YEAR_MIN/10)*10;d<=YEAR_MAX;d+=10) labels.push(d);
  const peak = Math.max(1,...counts.values());
  $('decades').innerHTML = labels.map((d,i)=>{
    const n=counts.get(d)||0, h= Math.max(2, 54*n/peak);
    return `<div class="decbar" data-decade="${d}" style="height:${h}px;opacity:${n ? 1 : .22}" title="${d}s: ${n} buildings; click to jump"></div>`;
  }).join('');
  $('decades').querySelectorAll('.decbar').forEach(el=>el.addEventListener('click',()=>{
    selectedYear = Math.min(YEAR_MAX,Number(el.dataset.decade)+9);
    $('yearSlider').value=selectedYear; stopPlay(); refresh();
  }));
}
function refresh(){
  $('year').textContent=selectedYear;
  $('modeNote').textContent=cumulative?'Built by selected year':'Built in this year';
  $('heightLabel').textContent=minHeight.toLocaleString()+' ft';
  shown=filtered();
  $('count').textContent=shown.length.toLocaleString();
  $('maxHeight').textContent=shown.length ? Math.round(Math.max(...shown.map(d=>d.height_ft))).toLocaleString() : '—';
  renderMap();renderRanks();renderDecades();
}
function setActive(id1,id2,first){$(id1).classList.toggle('active',first);$(id2).classList.toggle('active',!first);}
$('yearSlider').addEventListener('input',e=>{selectedYear=Number(e.target.value);stopPlay();refresh();});
$('heightSlider').addEventListener('input',e=>{minHeight=Number(e.target.value);refresh();});
$('scaleSelect').addEventListener('change',e=>{scale3D=Number(e.target.value);refresh();});
$('search').addEventListener('input',e=>{searchText=e.target.value.trim().toLowerCase();refresh();});
$('cumulative').addEventListener('click',()=>{cumulative=true;setActive('cumulative','exact',true);refresh();});
$('exact').addEventListener('click',()=>{cumulative=false;setActive('cumulative','exact',false);refresh();});
$('threeD').addEventListener('click',()=>{view3D=true;setActive('threeD','twoD',true);refresh();});
$('twoD').addEventListener('click',()=>{view3D=false;setActive('threeD','twoD',false);refresh();});
function stopPlay(){if(playback){clearInterval(playback);playback=null;}$('playButton').textContent='▶ Play skyline growth (+5 years / frame)';}
$('playButton').addEventListener('click',()=>{
  if(playback){stopPlay();return;}
  if(selectedYear>=YEAR_MAX) selectedYear=YEAR_MIN;
  cumulative=true;setActive('cumulative','exact',true);
  $('playButton').textContent='■ Pause timeline';
  playback=setInterval(()=>{
    selectedYear=Math.min(YEAR_MAX,selectedYear+5);
    $('yearSlider').value=selectedYear;refresh();
    if(selectedYear>=YEAR_MAX)stopPlay();
  },350); refresh();
});
$('exportCSV').addEventListener('click',()=>{
  const cols=['name','bin','year','height_ft','height_m','lat','lon'];
  const csv=[cols.join(','),...shown.map(d=>cols.map(k=>'"'+String(d[k]).replace(/"/g,'""')+'"').join(','))].join('\r\n');
  const blob=new Blob([csv],{type:'text/csv;charset=utf-8;'});
  const a=document.createElement('a');a.href=URL.createObjectURL(blob);a.download=`manhattan_buildings_${selectedYear}.csv`;a.click();
  setTimeout(()=>URL.revokeObjectURL(a.href),1000);
});
refresh();
</script></body></html>'''

In [ ]:
# Build a self-contained dashboard data bundle. Online JS and map tiles load in the browser.
records = buildings[["id", "name", "named", "bin", "year", "height_ft", "height_m", "lat", "lon"]].to_dict(orient="records")
js_data = json.dumps(records, ensure_ascii=False, separators=(",", ":"))
# Don't let an unusual building name terminate a <script> tag.
js_data = js_data.replace("<", "\\u003c").replace(">", "\\u003e").replace("&", "\\u0026")

html_doc = (HTML_TEMPLATE
    .replace("__DATA__", js_data)
    .replace("__MINYEAR__", str(EARLIEST_YEAR))
    .replace("__MAXYEAR__", str(LATEST_YEAR))
    .replace("__GENERATED__", date.today().isoformat()))

output_path = Path("manhattan_skyline_dashboard.html").resolve()
output_path.write_text(html_doc, encoding="utf-8")
print(f"Dashboard written to {output_path} ({len(html_doc)/1024:.0f} KiB)")
print("If the inline preview does not load, download/open the HTML in a regular browser.")

# srcdoc iframe gives Colab a self-contained browser context for its JavaScript.
iframe_srcdoc = html.escape(html_doc, quote=True)
display(HTML(
    f'<iframe title="Manhattan skyline dashboard" '
    f'style="display:block;width:100%;height:760px;border:1px solid #dce3e9;border-radius:12px" '
    f'sandbox="allow-scripts allow-same-origin allow-downloads" '
    f'srcdoc="{iframe_srcdoc}"></iframe>'
))

## Download the HTML and underlying building data

The downloadable HTML keeps all selected NYC building records in the file, so the slider, 2D/3D switch, bar chart, and filters work without rerunning Python. You **do need internet access** for the map tiles and deck.gl JavaScript library when you open it. The raw CSV is also useful for your own modeling or analysis.

Run the next cell to save both files and optionally download them from Colab.

In [ ]:
csv_path = Path("manhattan_tall_buildings.csv").resolve()
buildings.to_csv(csv_path, index=False)
print("CSV:", csv_path)
print("Interactive HTML:", output_path)

# To download from Google Colab, uncomment these two lines:
# from google.colab import files
# files.download(str(output_path))   # interactive 3D dashboard
# files.download(str(csv_path))      # original mapped building records

## Method, sources, and limitations

1. **Building footprints vs centroids.** NYC Open Data's [BUILDING_P](https://data.cityofnewyork.us/d/u9wf-3gbt) provides one mapped point per footprint along with `construction_year`, `height_roof`, and `bin`. It is a city building inventory, not a list of only famous towers. The official [geospatial metadata](https://github.com/CityOfNewYork/nyc-geo-metadata/blob/main/Metadata/Metadata_BuildingFootprints.md) describes the attributes.
2. **What counts as Manhattan?** The first digit of `bin` equals 1 for Manhattan. Keep feature code 2100 (actual building), `height_roof >= 100` ft for the downloaded base dataset, and valid construction year. The default browser threshold is 150 ft; increase it for only tall buildings.
3. **3D representation.** Each 3D pillar stands at the building centroid. The pillar's height is `height_roof × 0.3048` meters above its mapped site. These are **columns, not building-shaped footprints**. Its width is visually chosen for legibility and does not equal the building's footprint.
4. **Meaning of year.** `construction_year` is intended to indicate when construction was completed. Some years are approximate; many historic property dates reflect coarse assessments. “Built by 1930” includes only *buildings still in the current inventory*, not the full 1930 skyline.
5. **Caveats.** Roof height excludes spires and antennas; missing or unrecorded heights/years are not plotted; the inventory can contain imperfect identifiers and points. Named labels are sparse and not actively maintained. OSM/CARTO map imagery and JavaScript are loaded from public online services.

Map: [OpenStreetMap contributors](https://www.openstreetmap.org/copyright), [CARTO basemaps](https://carto.com/basemaps). Interactive 3D: [deck.gl](https://deck.gl/docs/api-reference/geo-layers/tile-layer). Data: NYC Office of Technology and Innovation / NYC Open Data.

**Extension idea:** If you eventually want realistic building-shaped 3D blocks, query the official [BUILDING footprint polygons](https://data.cityofnewyork.us/d/5zhs-2jue) rather than the `_P` point-centroid version, and render the polygon geometry with deck.gl `GeoJsonLayer` extrusions. The timeline and filters can remain the same.